# 206. ROME 风格知识编辑：rank-one update 与 locality 怎样验证？

> **面试问题：怎样用低秩更新改写一个事实，同时独立检查 rewrite、paraphrase generalization、无关事实 locality、冲突编辑与版本回滚？**

## 先给结论

这类题要把论文概念拆为可判定的数学/状态合同：方向从什么对比样本估计、解码如何保留合法候选、熵统计的概率空间是什么、权重编辑如何验证 rewrite/generalization/locality。教学实现用受控小向量，不代表真实模型安全、语言质量或跨领域泛化。

## 一手资料

- [ROME](https://arxiv.org/abs/2202.05262)
- [MEMIT](https://arxiv.org/abs/2210.07229)
- [Knowledge Editing Survey](https://arxiv.org/abs/2310.16218)

In [ ]:
notebook_contract = {"mode": "small-controlled-arrays", "oracle": "assertions", "production": "needs-evaluation-and-versioning"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "small-controlled-arrays"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：知识编辑要同时测 rewrite、generalization 与 locality

把某个 prompt 的输出改成目标答案并不等于完成知识编辑。至少要验证同一事实的改写 prompt 是否生效、无关事实是否保持、编辑可否被版本化回滚。ROME 的核心思想是对某层权重做局部低秩更新。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
weight = [[1.0, 0.0], [0.0, 1.0]]  # 执行本行的状态、计算或校验逻辑。
key = (1.0, 2.0)  # 执行本行的状态、计算或校验逻辑。
target = (4.0, -1.0)  # 执行本行的状态、计算或校验逻辑。
def matvec(matrix, vector):  # 执行本行的状态、计算或校验逻辑。
    return tuple(sum(row[index] * vector[index] for index in range(len(vector))) for row in matrix)  # 执行本行的状态、计算或校验逻辑。
assert matvec(weight, key) == key  # 执行本行的状态、计算或校验逻辑。
assert target != matvec(weight, key)  # 执行本行的状态、计算或校验逻辑。
assert len(weight) == len(key)  # 执行本行的状态、计算或校验逻辑。


## 2. rank-one update：使 W'k 精确命中新 target

对一个线性层，最小演示可取 `ΔW = (target - Wk) k^T / (k^T k)`。这在 k 非零时保证 `ΔW k` 恰为残差；真实 ROME 还涉及协方差、层选择、上下文和非线性网络，不能将此简化公式直接视为论文完整实现。


In [ ]:
def outer(left, right):  # 执行本行的状态、计算或校验逻辑。
    return [[a * b for b in right] for a in left]  # 执行本行的状态、计算或校验逻辑。
def rank_one_delta(weight, key, target):  # 执行本行的状态、计算或校验逻辑。
    current = matvec(weight, key)  # 执行本行的状态、计算或校验逻辑。
    residual = tuple(a - b for a, b in zip(target, current))  # 执行本行的状态、计算或校验逻辑。
    denominator = sum(value * value for value in key)  # 执行本行的状态、计算或校验逻辑。
    if denominator == 0:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("编辑 key 不能为零")  # 执行本行的状态、计算或校验逻辑。
    return [[value / denominator for value in row] for row in outer(residual, key)]  # 执行本行的状态、计算或校验逻辑。
delta = rank_one_delta(weight, key, target)  # 执行本行的状态、计算或校验逻辑。
assert delta == [[0.6, 1.2], [-0.6, -1.2]]  # 执行本行的状态、计算或校验逻辑。
assert len(delta) == 2  # 执行本行的状态、计算或校验逻辑。
assert len(delta[0]) == 2  # 执行本行的状态、计算或校验逻辑。


## 3. 提交：创建新权重版本而不是覆盖基线

生产编辑应创建带 lineage 的新 checkpoint/adapter 版本，便于灰度、审计和回滚。教学实现返回新矩阵；原矩阵保持不变是最基本的不变量，避免失败编辑污染全局基线。


In [ ]:
def add_matrix(left, right):  # 执行本行的状态、计算或校验逻辑。
    return [[a + b for a, b in zip(left_row, right_row)] for left_row, right_row in zip(left, right)]  # 执行本行的状态、计算或校验逻辑。
edited_weight = add_matrix(weight, delta)  # 执行本行的状态、计算或校验逻辑。
assert all(math.isclose(actual, expected) for actual, expected in zip(matvec(edited_weight, key), target))  # 执行本行的状态、计算或校验逻辑。
assert weight == [[1.0, 0.0], [0.0, 1.0]]  # 执行本行的状态、计算或校验逻辑。
assert edited_weight != weight  # 执行本行的状态、计算或校验逻辑。


## 4. rewrite：编辑 prompt 的目标输出必须由独立 verifier 判定

对于事实问答，可由 target token/字符串/任务 verifier 检查；对于开放文本，必须定义可验证属性而不是让同一模型给自己打分。这里使用精确向量 oracle，强调编辑成功与模型自述无关。


In [ ]:
def rewrite_success(weight, key, target, tolerance=1e-9):  # 执行本行的状态、计算或校验逻辑。
    output = matvec(weight, key)  # 执行本行的状态、计算或校验逻辑。
    return all(math.isclose(actual, expected, abs_tol=tolerance) for actual, expected in zip(output, target))  # 执行本行的状态、计算或校验逻辑。
assert rewrite_success(edited_weight, key, target)  # 执行本行的状态、计算或校验逻辑。
assert not rewrite_success(weight, key, target)  # 执行本行的状态、计算或校验逻辑。
assert rewrite_success(edited_weight, key, (4.0, -1.0))  # 执行本行的状态、计算或校验逻辑。


## 5. locality：与 key 正交的输入在 rank-one 简化中保持不变

更新只沿 key 外积方向改变，因此与 key 正交的 query 理论上不受影响。这是教学的 locality oracle；真实 Transformer 的非线性和层间耦合会让 locality 更复杂，必须用无关事实集评测。


In [ ]:
unrelated = (2.0, -1.0)  # 执行本行的状态、计算或校验逻辑。
before = matvec(weight, unrelated)  # 执行本行的状态、计算或校验逻辑。
after = matvec(edited_weight, unrelated)  # 执行本行的状态、计算或校验逻辑。
assert sum(a * b for a, b in zip(key, unrelated)) == 0.0  # 执行本行的状态、计算或校验逻辑。
assert before == after  # 执行本行的状态、计算或校验逻辑。
assert before == unrelated  # 执行本行的状态、计算或校验逻辑。


## 6. generalization：同一事实的改写 key 不保证自动成功

不同 prompt 的内部 key 通常不完全相同，因此单点编辑可能只记住触发模板。这里构造接近 key 的 paraphrase 并显式测量误差，说明 generalization 是必须独立报告的指标。


In [ ]:
paraphrase_key = (1.0, 1.8)  # 执行本行的状态、计算或校验逻辑。
paraphrase_output = matvec(edited_weight, paraphrase_key)  # 执行本行的状态、计算或校验逻辑。
paraphrase_error = math.sqrt(sum((actual - expected) ** 2 for actual, expected in zip(paraphrase_output, target)))  # 执行本行的状态、计算或校验逻辑。
assert paraphrase_error > 0  # 执行本行的状态、计算或校验逻辑。
assert paraphrase_error < math.sqrt(sum((actual - expected) ** 2 for actual, expected in zip(matvec(weight, paraphrase_key), target)))  # 执行本行的状态、计算或校验逻辑。
assert paraphrase_output != target  # 执行本行的状态、计算或校验逻辑。


## 7. 失败分支：冲突编辑、零 key 与版本不匹配必须受控

重复编辑同一事实、编辑互相冲突、或在错误 base checkpoint 上应用 delta 都可能破坏模型。至少要比较 base fingerprint、事实 id 和冲突策略；不能把 delta 文件当作可跨模型复制的普通补丁。


In [ ]:
def compatible_edit(edit, runtime_base):  # 执行本行的状态、计算或校验逻辑。
    return edit["base"] == runtime_base and edit["fact_id"] != ""  # 执行本行的状态、计算或校验逻辑。
edit_meta = {"base": "model-v1", "fact_id": "capital-france", "method": "rank-one"}  # 执行本行的状态、计算或校验逻辑。
assert compatible_edit(edit_meta, "model-v1")  # 执行本行的状态、计算或校验逻辑。
assert not compatible_edit(edit_meta, "model-v2")  # 执行本行的状态、计算或校验逻辑。
assert not compatible_edit({**edit_meta, "fact_id": ""}, "model-v1")  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：编辑 delta、评测集与回滚目标一起记录

安全的发布单元包含 base、delta hash、事实/来源、rewrite-generalization-locality 分数、批准人和 rollback target。生产上还需要访问控制、灰度监控、触发条件和可撤销的 serving 路由。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"base": "model-v1", "method": "rank-one", "fact": "capital-france", "rewrite": True, "locality": before == after, "rollback": "model-v1"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["rewrite"] is True  # 执行本行的状态、计算或校验逻辑。
assert artifact["locality"] is True  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明目标与状态，再给出核心公式、失败反例、独立评测和版本化制品。不要把对一个合成向量/几个候选的断言通过，误说成真实大模型上已经可靠、无偏或安全。
